# Milestones 3-7 - backtest, baseline ladder, LightGBM, official comparison

**Runtime → Run all.** About 35 minutes on Colab's CPU runtime. Nothing to decide along the way.

What it does, in order:

1. Rebuilds the demand Parquet from IESO (about a minute) unless it is already there
2. Runs the test suite - above all the leakage guard, which proves no feature
   for day D can see data published after the 12:30 origin. If that fails, the
   notebook stops: every result after it would be optimistic.
3. Runs `python -m src.evaluate.run`: the whole ladder over 2024-2025, then the
   same models against IESO's own day-ahead forecast on every day the harvester
   has archived
4. Shows the headline and figures, and hands you a zip of the outputs

The logic is all in `src/`; this notebook only drives it, so a Colab run and a
local run are the same code.

**If the weather download fails** (Track B needs Environment Canada data),
Track A and everything else still finish, and the report says Track B did not
run and why.

## 1. Setup

In [1]:
# polars is upgraded explicitly: pip leaves an older preinstalled copy in place
# otherwise, and this code was tested on polars 2.x.
!pip install -q -U polars
!pip install -q duckdb holidays lightgbm statsforecast pytest

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 876.6/876.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 501.8/501.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.7/348.7 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 4.3 MB/s eta 0:00:00


In [2]:
USE_DRIVE = False   # True keeps downloads in Drive between sessions (asks for Drive access)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

In [3]:
import os, pathlib, subprocess

REPO_URL = "https://github.com/ethantosc/ieso-demand-forecast.git"
REPO_DIR = pathlib.Path("/content/ieso-demand-forecast")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

5011a08 Milestones 3-8: backtest, baseline ladder, LightGBM tracks A/B, official comparison


Only the rebuildable directories are redirected. `data/raw/forecasts/` - the
harvested IESO forecast archive - stays exactly where the clone put it.

In [4]:
import shutil

DATA_ROOT = pathlib.Path("/content/drive/MyDrive/ieso-demand-forecast-data" if USE_DRIVE
                         else "/content/ieso-demand-forecast-data")
for sub in ("raw/demand", "raw/weather", "staging", "curated"):
    (DATA_ROOT / sub).mkdir(parents=True, exist_ok=True)
    link = pathlib.Path("data") / sub
    link.parent.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        if [p for p in link.rglob("*") if p.suffix.lower() in {".csv", ".parquet"}]:
            raise SystemExit(f"data/{sub} holds data files. Decide before re-running.")
        shutil.rmtree(link)
    link.symlink_to(DATA_ROOT / sub)
print("store:", DATA_ROOT, "| forecast archive files:", len(list(pathlib.Path("data/raw/forecasts").rglob("*.xml"))))

store: /content/ieso-demand-forecast-data | forecast archive files: 127


In [5]:
if pathlib.Path("data/staging/demand_hourly.parquet").exists():
    print("staging Parquet already present")
else:
    subprocess.run(["python", "-m", "src.ingest.fetch_demand"], check=True)
    subprocess.run(["python", "-m", "src.ingest.demand"], check=True)

## 2. Tests

Eleven tests. The ones that matter most corrupt every observation at or after the
day-ahead cutoff and check that no feature for that day changes, plus a control
proving the check is not vacuous. A failure here stops the notebook.

In [6]:
result = subprocess.run(["python", "-m", "pytest", "tests", "-q"], capture_output=True, text=True)
print(result.stdout[-2000:], result.stderr[-2000:])
if result.returncode != 0:
    raise SystemExit("Tests failed - do not trust any result below. Send the output above.")

...........                                                              [100%]
11 passed in 1.01s
 


## 3. Run

Progress prints as it goes: weather download, then each model per window. The
classical models refit at every origin, so they take the longest.

In [ ]:
!python -m src.evaluate.run

[04:29:33] loading inputs
[04:29:33] staging 214,488 rows; blackout exclusion 6 days (2003-08-14 .. 2003-08-19)
weather files: {'downloaded': 874, 'cached': 1}
wrote data/staging/weather_hourly.parquet  208,585 hours  2003-01-01 00:00:00 .. 2026-10-18 23:00:00
shape: (24, 5)
┌─────┬─────┬─────┬─────┬─────┐
│ yea ┆ tor ┆ ott ┆ lon ┆ tem │
│ r   ┆ ont ┆ awa ┆ don ┆ p   │
│ --- ┆ o   ┆ --- ┆ --- ┆ --- │
│ i32 ┆ --- ┆ f64 ┆ f64 ┆ f64 │
│     ┆ f64 ┆     ┆     ┆     │
╞═════╪═════╪═════╪═════╪═════╡
│ 200 ┆ 1.0 ┆ 1.0 ┆ 0.9 ┆ 1.0 │
│ 3   ┆     ┆     ┆ 99  ┆     │
│ 200 ┆ 1.0 ┆ 1.0 ┆ 1.0 ┆ 1.0 │
│ 4   ┆     ┆     ┆     ┆     │
│ 200 ┆ 1.0 ┆ 1.0 ┆ 1.0 ┆ 1.0 │
│ 5   ┆     ┆     ┆     ┆     │
│ 200 ┆ 1.0 ┆ 1.0 ┆ 1.0 ┆ 1.0 │
│ 6   ┆     ┆     ┆     ┆     │
│ 200 ┆ 1.0 ┆ 1.0 ┆ 1.0 ┆ 1.0 │
│ 7   ┆     ┆     ┆     ┆     │
│ …   ┆ …   ┆ …   ┆ …   ┆ …   │
│ 202 ┆ 0.9 ┆ 0.9 ┆ 0.9 ┆ 1.0 │
│ 2   ┆ 99  ┆ 99  ┆ 99  ┆     │
│ 202 ┆ 1.0 ┆ 1.0 ┆ 0.9 ┆ 1.0 │
│ 3   ┆     ┆     ┆ 99  ┆     │
│ 202 ┆ 1.0 ┆ 1.0 ┆ 

## 4. Headline

In [ ]:
from IPython.display import Image, Markdown, display

report = pathlib.Path("reports/findings_auto.md").read_text()
start = report.index("## 1. Headline")
end = report.index("\n## ", start + 5)
display(Markdown(report[start:end]))
for name in ["backtest_ladder", "backtest_by_hour", "backtest_residuals",
             "official_ladder", "official_daily", "official_by_hour"]:
    path = pathlib.Path(f"reports/figures/{name}.png")
    if path.exists():
        display(Image(filename=str(path)))

## 5. Take the outputs with you

The clone disappears with the session. This zips `reports/` in repo layout.
On your Mac: unzip, then drag the `reports` folder onto GitHub's
**Add file → Upload files** page at the repo root. GitHub keeps the folder
structure and replaces older copies.

Then send `reports/findings_auto.md` back to Claude, which writes the
interpretation in `findings.md` from it.

In [ ]:
out = pathlib.Path("/content/ieso_results")
if out.exists():
    shutil.rmtree(out)
(out / "reports").mkdir(parents=True)
shutil.copy("reports/findings_auto.md", out / "reports" / "findings_auto.md")
for sub in ("results", "figures"):
    shutil.copytree(f"reports/{sub}", out / "reports" / sub)
archive = shutil.make_archive("/content/ieso_results", "zip", out)
if USE_DRIVE:
    shutil.copy(archive, DATA_ROOT / "ieso_results.zip")
    print("copied to", DATA_ROOT / "ieso_results.zip")
from google.colab import files
files.download(archive)